In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# =====================================================
# 1. CONFIGURATION (CHANGE ONLY THIS PART)
# =====================================================

DATA_PATH = r"C:\Users\Sam\Desktop\ML\task\Data.xlsx"
sheet_name = "Data_after_KFold_DTC"
CONFIG = {
    "optimizer": "KOA",  # changed from BOA
    "population": 25,
    "iterations": 200,
    "cv": 5,
    "random_state": 42
}

# =====================================================
# 2. LOAD DATA
# =====================================================

df = pd.read_excel(DATA_PATH, sheet_name=sheet_name)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=CONFIG["random_state"], stratify=y
)

MODEL = {
    "name": "SVC",
    "builder": SVC,
    "bounds": {
        "C": (0.1, 10.0, float),
        "gamma": (0.001, 1.0, float)
    }
}


# =====================================================
# 3. HELPER FUNCTIONS
# =====================================================

def bounds_to_arrays(bounds):
    lb, ub, cast = [], [], []
    for v in bounds.values():
        lb.append(v[0])
        ub.append(v[1])
        cast.append(v[2])
    return np.array(lb), np.array(ub), cast

def decode_params(vec, bounds, cast):
    decoded = {}
    for i, k in enumerate(bounds.keys()):
        decoded[k] = cast[i](vec[i])
    return decoded


def make_objective(model_builder, bounds, cast):
    def objective(vec):
        params = decode_params(vec, bounds, cast)

        model = model_builder(
            **params,
            kernel="rbf",
            random_state=CONFIG["random_state"],
            cache_size=500
        )

        score = cross_val_score(
            model,
            X_train,
            y_train,
            cv=CONFIG["cv"],
            scoring="accuracy",
            n_jobs=-1
        ).mean()

        return -score  # KOA minimizes
    return objective


# =====================================================
# 4. KANGAROO OPTIMIZATION ALGORITHM (KOA)
# =====================================================

def KOA(objective, lb, ub, N, T, cast):
    start = time.time()
    D = len(lb)

    # --- Initialize population ---
    pop = lb + np.random.rand(N, D) * (ub - lb)
    fit = np.array([objective(pop[i]) for i in range(N)])

    best_idx = np.argmin(fit)
    best = pop[best_idx].copy()
    best_fit = fit[best_idx]

    convergence = []
    log = []

    for t in range(T):
        # KOA-inspired movement (jump towards best and random exploration)
        for i in range(N):
            r1, r2 = np.random.rand(D), np.random.rand(D)
            jump_to_best = r1 * (best - pop[i])
            random_jump = r2 * (np.mean(pop, axis=0) - pop[i])
            candidate = pop[i] + jump_to_best + random_jump
            candidate = np.clip(candidate, lb, ub)

            f = objective(candidate)

            # Update if improvement
            if f < fit[i]:
                pop[i] = candidate
                fit[i] = f
                if f < best_fit:
                    best, best_fit = candidate.copy(), f

        # --- Record convergence per iteration ---
        convergence.append(-best_fit)

        # --- Log only the best solution at this iteration ---
        best_decoded = decode_params(best, MODEL["bounds"], cast)
        log.append([t + 1] + [best_decoded[k] for k in MODEL["bounds"].keys()] + [-best_fit])

        # --- Print per iteration ---
        print(
            f"Iter {t+1:03d}, Best = "
            + ", ".join(f"{k}={v}" for k, v in best_decoded.items())
            + f", Acc = {-best_fit:.4f}"
        )

    runtime = time.time() - start
    return decode_params(best, MODEL["bounds"], cast), -best_fit, convergence, runtime, log

# =====================================================
# 5. RUN OPTIMIZATION
# =====================================================

lb, ub, cast = bounds_to_arrays(MODEL["bounds"])
objective = make_objective(MODEL["builder"], MODEL["bounds"], cast)

best, best_acc, convergence, runtime, log = KOA(
    objective, lb, ub, CONFIG["population"], CONFIG["iterations"], cast
)

# =====================================================
# 6. FINAL MODEL & REPORT
# =====================================================

best_params = best
final_model = MODEL["builder"](
    **best_params,
    kernel="rbf",
    random_state=CONFIG["random_state"],
    cache_size=500
)

final_model.fit(X_train, y_train)

test_acc = accuracy_score(y_test, final_model.predict(X_test))

# =====================================================
# 7. TABLES (EXCEL / PAPER READY)
# =====================================================

# Full iterations log
iter_cols = ["iteration"] + list(MODEL["bounds"].keys()) + ["best_cv_accuracy"]
iterations_df = pd.DataFrame(log, columns=iter_cols)

# Convergence per iteration
convergence_df = pd.DataFrame({"best_accuracy": convergence})

# Summary
summary_df = pd.DataFrame([{
    "Model": MODEL["name"],
    "Optimizer": CONFIG["optimizer"],  # shows KOA
    "Runtime_sec": runtime
}])

# Best hyperparameters table
best_params_df = pd.DataFrame({
    "parameters": list(best_params.keys()),
    "values": list(best_params.values())
})

# =========================
# PRINT RESULTS
# =========================
print("\n✅ Best Hyperparameters Table:")
print(best_params_df)

print("\n✅ Summary Table:")
print(summary_df)

print("\n✅ Iterations Log Preview:")
print(iterations_df.head(10))  # show first 10 rows

print("\n✅ Convergence Preview:")
print(convergence_df.head(10))


Iter 001, Best = C=3.952580608454538, gamma=0.4841004046175491, Acc = 0.8497
Iter 002, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 003, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 004, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 005, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 006, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 007, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 008, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 009, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 010, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 011, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 012, Best = C=4.290335398152547, gamma=0.47815147173980055, Acc = 0.8508
Iter 013, Best = C=4.290335398152547, gamma=0.47815147173980055, 

In [2]:
iterations_df.to_clipboard(index=False)

In [3]:
convergence_df.to_clipboard(index=False)

In [4]:

summary_df.to_clipboard(index=False)

In [5]:
best_params_df.to_clipboard(index=False)